In [1]:
import torch
from tqdm.autonotebook import tqdm

from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.model.mlp import StackedResidualMLPConfig
from src.model.time_conditioning import (
    CategoricalConditioningConfig,
    TimeConditioningConfig,
)
from src.monitoring.utils import pca_project, scatterplot
from src.priors.anchored import AnchoredGaussianScaleMixturePriorConfig
from src.stochastic_chart_transport.critic import CriticLossConfig
from src.stochastic_chart_transport.fibers import FlatFiberPacking
from src.stochastic_chart_transport.model import ChartTransportModelConfig
from src.stochastic_chart_transport.reconstruction import (
    AnchorLossConfig,
    ChartPretrainConfig,
    ReconstructionLossConfig,
)
from src.stochastic_chart_transport.study import (
    StochasticChartTransportStudyConfig,
    StochasticChartTransportStudyState,
)
from src.stochastic_chart_transport.transport import StochasticChartTransportLossConfig

device = "cuda:1"

/tmp/ipykernel_1813246/3741297964.py:2: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
num_modes = 8
batch_size = 4096
ambient_dimension = 3
fiber_dimension = ambient_dimension
hidden_dim = 1024
hidden_dims = [hidden_dim] * 4

In [3]:
mc = ChartTransportModelConfig(
    encoder=StackedResidualMLPConfig.initialize(
        layer_dims=[ambient_dimension + fiber_dimension]
        + hidden_dims
        + [ambient_dimension]
    ),
    decoder=StackedResidualMLPConfig.initialize(
        layer_dims=[ambient_dimension]
        + hidden_dims
        + [ambient_dimension + fiber_dimension]
    ),
    critic=StackedResidualMLPConfig.initialize(
        layer_dims=[ambient_dimension] + hidden_dims + [ambient_dimension],
        time_conditioning_config=TimeConditioningConfig(
            min_t_lambda=1e-3,
            max_t_lambda=1.0,
            condition_dim=hidden_dim,
        ),
        cat_conditioning_config=CategoricalConditioningConfig(
            num_classes=2,
            condition_dim=hidden_dim,
        ),
    ),
    chart_lr=3e-4,
    critic_lr=3e-4,
    grad_clip_norm=2.0,
)

pretrain_config = ChartPretrainConfig(
    reconstruction_config=ReconstructionLossConfig(
        huber_delta=10.0,
        data_reconstruction_weight=1.0,
        prior_reconstruction_weight=1.0,
        stochastic_reconstruction_divider=10,
    ),
    # These have lower weight because they're "prior" configs
    anchor_config=AnchorLossConfig(
        latent_norm_weight=1e-2, latent_zero_mean_weight=0.1
    ),
)

transport_config = StochasticChartTransportLossConfig(
    transport_step_multiplier=0.05,
    transport_step_cap=0.05,
    decoder_transport_weight=1.0,
    encoder_transport_weight=1.0,
    decoder_huber_delta=5.0,
    antipodal_estimate=True,
    t_range=(0.03, 0.97),
    num_time_samples=5,
)

In [4]:
study_config = StochasticChartTransportStudyConfig(
    data=MultimodalGaussianDataConfig.initialize(
        num_modes=8, mode_std=0.05, ambient_dimension=ambient_dimension, offset=0
    ),
    prior=AnchoredGaussianScaleMixturePriorConfig.initialize(
        latent_shape=[ambient_dimension], precision=3.0
    ),
    model=mc,
    pretrain=pretrain_config,
    critic=CriticLossConfig(huber_delta=10.0, weight=1.0, t_min=0.01, t_max=0.99),
    transport=transport_config,
    fiber_packing=FlatFiberPacking(fiber_ndims=fiber_dimension),
)
display(study_config.visualize())
state = StochasticChartTransportStudyState.initialize(
    config=study_config, device=torch.device(device)
)

## Chart pretrain

In [5]:
pbar = tqdm(range(1000))

for p in pbar:
    data = state.config.data.sample_unconditional(batch_size=batch_size).to(device)

    loss = state.compute_chart_only_loss(data=data, compute_anchor_loss=True)

    total_loss = loss.sum()
    total_loss.backward()
    state.step_and_zero_grad()
    pbar.set_postfix({"loss": total_loss.item()})

  0%|          | 0/1000 [00:00<?, ?it/s]

In [10]:
visualize_batch_size_per_mode = 1024
canonical_samples = {
    str(j): study_config.data.sample_class(
        mode_id=j, batch_size=visualize_batch_size_per_mode
    ).to(device)
    for j in range(num_modes)
}


def monitor(state):
    prior = state.prior_config.sample(batch_size=visualize_batch_size_per_mode).to(
        device
    )
    with torch.no_grad():
        model_sample = state.decode(prior)[0]
        fiber = torch.randn_like(model_sample)
        model_latent = state.encode(data=model_sample, fiber=fiber)
    display(
        scatterplot(
            pca_project(
                {
                    k: state.encode(
                        data=v,
                        fiber=torch.randn_like(v),
                    )
                    for k, v in canonical_samples.items()
                }
                | {"model": model_latent},
                pca_dim=3,
            ),
        )
    )
    display(scatterplot(pca_project({"model": model_sample}, pca_dim=3)))


monitor(state)

## Critic score matching

In [11]:
pbar = tqdm(range(1_000))

for p in pbar:
    data = state.config.data.sample_unconditional(batch_size=batch_size).to(device)

    loss = state.compute_critic_only_loss(data=data)
    loss = loss.sum()
    loss.backward()
    state.step_and_zero_grad()

    pbar.set_postfix({"loss": loss.item()})


  0%|          | 0/1000 [00:00<?, ?it/s]

## Integrated transport

In [ ]:
pbar = tqdm(range(1_000))

for p in pbar:
    data = state.config.data.sample_unconditional(batch_size=batch_size).to(device)

    losses = state.compute_integrated_loss(
        data=data,
        compute_transport_loss=(p % 5 == 0),
    )

    total_loss = losses.sum()
    total_loss.backward()
    state.step_and_zero_grad()

    pbar.set_postfix({"loss": total_loss.item()})
    if p % 100 == 0:
        monitor(state)

  0%|          | 0/1000 [00:00<?, ?it/s]